# New Approach Methods (NAMs) in Toxicology
### Industry-Standard Tutorial — EPA · FDA · OECD · ICH Aligned

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)  
**Repo:** `computational-science-tutorials`

---

## What are New Approach Methods?

NAMs are any technology, methodology, approach, or combination that can provide information on chemical hazard and risk potential **without using intact animals** (EPA 2018).

```
Traditional toxicology                    NAM-based toxicology
─────────────────────                     ────────────────────────────────
Rat 28-day repeat-dose                →   QSAR + IVIVE + organ-on-chip
Rabbit eye irritation (Draize)        →   EpiOcular 3D model + QSAR
LD50 rat oral                         →   Cytotoxicity + QSAR + PBPK
ICH S7B hERG-only cardiac            →   CiPA multi-channel + hiPS-CM
Carcinogenicity (2-year rat/mouse)    →   Genotoxicity battery + TGx-DDI
Developmental toxicity (rat)          →   zebrafish + WESI + GeSi
```

### Why NAMs matter RIGHT NOW

| Driver | Details |
|--------|---------|
| **EPA TSCA 2016** | Mandates replacement of vertebrate tests where scientifically valid |
| **FDA Modernization Act 2.0 (2022)** | Explicitly allows NAM data to replace animal data for IND submissions |
| **EU REACH** | NAM data package accepted for non-testing DNEL derivation |
| **ICH M7(R2) 2023** | In silico models for genotoxic impurity assessment (Class 1–3) |
| **OECD TG 497 (2023)** | Defined approaches for skin sensitisation — first OECD-validated NAM DA |

## What you will learn

| Section | NAM Type | Regulatory context |
|---------|----------|--------------------|
| 1. QSAR & in silico | Computational prediction | ICH M7, OECD QSAR principles |
| 2. Structural alerts | SMARTS-based screening | ICH M7, Derek Nexus logic |
| 3. Read-across | Category formation | OECD RAAF, IUCLID |
| 4. IVIVE | In vitro → in vivo extrapolation | EPA HTTK, EPA OPP |
| 5. Skin sensitisation DA | Defined approach (3-method) | OECD TG 442C/D/E, TG 497 |
| 6. CiPA cardiac safety | Multi-ion channel assay | FDA/CIPA initiative, ICH E14/S7B |
| 7. PBPK-NAM | Dosimetry + IVIVE integration | EPA, FDA, EFSA PBPK guidelines |
| 8. AOP framework | Adverse outcome pathway | OECD AOP-Wiki |
| 9. Integrated approaches | IATA / weight of evidence | OECD GD 255, ECHA R.7 |
| 10. Regulatory submission | Report generation | FDA, EPA, ECHA format |

---
## Section 1 — QSAR and In Silico Prediction

QSAR (Quantitative Structure-Activity Relationship) models are the most widely used NAMs. They are fully accepted by ICH M7(R2) for genotoxic impurity classification and OECD principles govern their regulatory validity.

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
# !pip install rdkit scikit-learn xgboost shap matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')
import json
from dataclasses import dataclass, field
from typing import Optional

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED, DataStructs
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

print("Imports OK ✓")

In [ ]:
# ── 1.1 The five OECD principles for regulatory QSAR ─────────────────────────
# A QSAR model is only regulatorily acceptable if it satisfies all 5 principles.
# (OECD 2004, updated in QMRF reports and OECD GD 69)

OECD_PRINCIPLES = {
    1: "Defined endpoint — unambiguous, measured under standard conditions",
    2: "Unambiguous algorithm — model is reproducible and fully described",
    3: "Defined domain of applicability (AD) — compounds outside AD must be flagged",
    4: "Appropriate measures of goodness-of-fit, robustness, and predictivity",
    5: "Mechanistic interpretation where possible",
}
print("OECD 5 Principles for Regulatory QSAR:")
for i, text in OECD_PRINCIPLES.items():
    print(f"  {i}. {text}")

# ── 1.2 Build a regulatory-grade QSAR model ───────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix
from collections import defaultdict

def smiles_to_ecfp4(smiles: str, n_bits: int = 2048) -> np.ndarray | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
    return np.array(fp, dtype=np.uint8)

# Simulate Ames mutagenicity dataset (replace with Hansen 2009 or ChemBL data)
np.random.seed(42)
n_compounds = 800
smiles_templates = [
    "CC(=O)Oc1ccccc1C(=O)O",    # aspirin-like: inactive
    "CC(=O)Nc1ccc(O)cc1",        # paracetamol-like: inactive
    "Cc1ccc(N)cc1",               # aromatic amine: potentially active
    "O=C1NC(=O)c2ccccc21",       # phthalimide: varies
    "c1ccc2[nH]ccc2c1",          # indole: varies
    "O=Nn1ccc2ccccc21",          # nitroindole: active alert
    "NCc1ccccc1",                 # benzylamine: varies
    "O=[N+]([O-])c1ccccc1",      # nitrobenzene: active
]

dataset = []
for i in range(n_compounds):
    smi = smiles_templates[i % len(smiles_templates)]
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mw   = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        # Simulated Ames label (correlated with alerts)
        has_nitro = mol.HasSubstructMatch(Chem.MolFromSmarts("[N+](=O)[O-]"))
        has_ar_amine = mol.HasSubstructMatch(Chem.MolFromSmarts("[NH2]c"))
        prob_active = 0.1 + 0.4*int(has_nitro) + 0.3*int(has_ar_amine)
        label = int(np.random.rand() < prob_active)
        dataset.append({"smiles": smi, "label": label})

df = pd.DataFrame(dataset)
print(f"\nDataset: {len(df)} compounds")
print(f"  Actives (mutagenic):  {df.label.sum()} ({df.label.mean()*100:.0f}%)")
print(f"  Inactives:            {(~df.label.astype(bool)).sum()}")

# Build feature matrix
fps = np.array([smiles_to_ecfp4(s) for s in df.smiles if smiles_to_ecfp4(s) is not None])
y   = df.label.values[:len(fps)]
print(f"  Feature matrix: {fps.shape}")

In [ ]:
# ── 1.3 Cross-validated performance (regulatory requirement) ──────────────────
# ICH M7 requires: sensitivity ≥ 90%, specificity reported, balanced dataset

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=42)

# Stratified 5-fold cross-validation (OECD GD 69 minimum standard)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_prob_cv = cross_val_predict(rf, fps, y, cv=cv, method='predict_proba')[:, 1]
y_pred_cv = (y_prob_cv >= 0.5).astype(int)

# Regulatory metrics
cm = confusion_matrix(y, y_pred_cv)
tn, fp_n, fn, tp = cm.ravel()
sensitivity  = tp / (tp + fn)   # recall for positives — ICH M7: must be ≥ 90%
specificity  = tn / (tn + fp_n)
ppv          = tp / (tp + fp_n)  # precision
npv          = tn / (tn + fn)
auc          = roc_auc_score(y, y_prob_cv)
mcc          = matthews_corrcoef(y, y_pred_cv)  # balanced metric, ICH favoured

print("5-fold CV Performance (OECD GD 69 format):")
print(f"  Sensitivity (recall) : {sensitivity:.3f}   ← ICH M7 threshold: ≥ 0.90")
print(f"  Specificity           : {specificity:.3f}")
print(f"  PPV (precision)       : {ppv:.3f}")
print(f"  NPV                   : {npv:.3f}")
print(f"  ROC-AUC               : {auc:.3f}")
print(f"  MCC                   : {mcc:.3f}")
print(f"\nConfusion matrix:")
print(f"  TN={tn}  FP={fp_n}")
print(f"  FN={fn}  TP={tp}")

# ICH M7 interpretation
if sensitivity >= 0.90:
    print("\n  ✓ Meets ICH M7(R2) sensitivity requirement (≥ 90%)")
else:
    print(f"\n  ✗ Below ICH M7(R2) sensitivity threshold (got {sensitivity:.2%}, need 90%)")
    print("    → Requires complementary expert review or structural alert screening")

# Fit final model on all data for AD calculation
rf.fit(fps, y)
print("\nFinal model trained ✓")

In [ ]:
# ── 1.4 Applicability domain (OECD Principle 3) ──────────────────────────────
# The AD defines which compounds the model can reliably predict.
# Compounds outside the AD must be flagged — not just silently predicted.

class TanimotoApplicabilityDomain:
    """
    Tanimoto-based applicability domain (Netzeva 2005).

    A test compound is within the AD if its similarity to at least
    one training compound exceeds a threshold (z-score method).

    This is the most commonly used AD method in regulatory submissions.
    """
    def __init__(self, threshold: float = None, percentile: float = 5.0):
        self.training_fps   = None
        self.threshold      = threshold
        self.percentile     = percentile  # flag bottom X% as outside AD

    def fit(self, fps: np.ndarray):
        self.training_fps = fps.astype(np.float32)
        # Compute pairwise Tanimoto within training set
        ixj   = fps @ fps.T
        sx    = fps.sum(axis=1, keepdims=True)
        union = sx + sx.T - ixj
        sim   = np.where(union > 0, ixj/union.astype(float), 0.0)
        np.fill_diagonal(sim, 0)  # exclude self-similarity
        max_sim = sim.max(axis=1)
        # Threshold: below this percentile → outside AD
        if self.threshold is None:
            self.threshold = np.percentile(max_sim, self.percentile)
        return self

    def predict_ad(self, test_fps: np.ndarray) -> np.ndarray:
        """Return bool array: True = within AD."""        test = test_fps.astype(np.float32)
        train = self.training_fps
        ixj   = test @ train.T
        st    = test.sum(axis=1, keepdims=True)
        sv    = train.sum(axis=1, keepdims=True).T
        union = st + sv - ixj
        sim   = np.where(union > 0, ixj/union, 0.0)
        max_sim = sim.max(axis=1)
        return max_sim >= self.threshold

    def get_max_similarity(self, test_fps: np.ndarray) -> np.ndarray:
        """Return max Tanimoto to training set per compound."""        test  = test_fps.astype(np.float32)
        train = self.training_fps
        ixj   = test @ train.T
        st    = test.sum(axis=1, keepdims=True)
        sv    = train.sum(axis=1, keepdims=True).T
        union = st + sv - ixj
        sim   = np.where(union > 0, ixj/union, 0.0)
        return sim.max(axis=1)

ad = TanimotoApplicabilityDomain(percentile=5.0)
ad.fit(fps)
ad_mask = ad.predict_ad(fps)
max_sim = ad.get_max_similarity(fps)

print(f"Applicability Domain (Tanimoto, threshold={ad.threshold:.3f}):")
print(f"  Compounds within AD: {ad_mask.sum()}/{len(fps)}  ({ad_mask.mean()*100:.0f}%)")
print(f"  Mean max-Tc (training): {max_sim.mean():.3f} ± {max_sim.std():.3f}")

# Test compound assessment
test_smi = "O=[N+]([O-])c1ccccc1"  # nitrobenzene
test_fp  = smiles_to_ecfp4(test_smi)
if test_fp is not None:
    test_in_ad  = ad.predict_ad(test_fp.reshape(1,-1))[0]
    test_max_tc = ad.get_max_similarity(test_fp.reshape(1,-1))[0]
    test_pred   = rf.predict_proba(test_fp.reshape(1,-1))[0,1]
    print(f"\nTest compound: {test_smi}")
    print(f"  Within AD: {test_in_ad}  (max Tc = {test_max_tc:.3f})")
    print(f"  Predicted probability (mutagenic): {test_pred:.3f}")
    print(f"  QSAR call: {'MUTAGENIC' if test_pred >= 0.5 else 'Non-mutagenic'}")
    if not test_in_ad:
        print("  ⚠ OUTSIDE AD — prediction requires expert review")

---
## Section 2 — Structural Alert Screening (ICH M7 / Derek Logic)

Structural alerts (SA) are SMARTS patterns encoding known toxicophore fragments. ICH M7(R2) requires TWO complementary in silico methodologies: one rule-based (SA) and one statistical (QSAR).

In [ ]:
# ── 2.1 ICH M7 structural alert library ──────────────────────────────────────
# Curated SMARTS for DNA-reactive (mutagenic) and genotoxic alerts.
# Based on: ICH M7(R2), Kazius/Bursi dataset, DEREK Nexus, Sarah Nexus

ICH_M7_ALERTS = {
    # Class 1 alerts — known human mutagens (avoid entirely)
    "Nitrosamine":              "[N;!$(N=O)]-N=O",
    "N-Nitroso_secondary":      "[#6][N;H0]([#6])N=O",
    "Nitrosamide":              "C(=O)N([#6])N=O",
    "Diazonium":                "[#6][N+]#N",

    # Class 2 alerts — known animal mutagens
    "Aromatic_nitro":           "c[N+](=O)[O-]",
    "Aliphatic_nitro":          "C[N+](=O)[O-]",
    "Aromatic_primary_amine":   "[NH2]c",
    "N_N_azoxy":                "[N;R0]=[N;R0]",
    "Epoxide":                  "[C;R0]1OC1",
    "Aziridine":                "C1CN1",
    "Alpha_beta_unsaturated_CO": "C=CC=O",
    "Michael_acceptor":         "[$(C=CC=O),$(C=CSR),$(C=CN)]",

    # Class 3 alerts — structural concern, less well characterised
    "Aromatic_azo":             "c-N=N-c",
    "Hydroxamic_acid":          "C(=O)NO",
    "Primary_alkyl_halide_Cl":  "[Cl][CX4][CX4]",
    "Primary_alkyl_halide_Br":  "[Br][CX4]",
    "Carbamimidoyl_nitrogen":   "[NX3][C]=[NX2]",
    "Propiolactone":            "C1CC(=O)O1",
    "Sultone":                  "C1CCS(=O)(=O)O1",

    # Alerting metabolites (reactive metabolite precursors)
    "Furan":                    "c1ccoc1",
    "Thiophene_metabolite":     "c1ccsc1",
    "Hydrazine":                "[NH2]N",
    "Alkenyl_halide":           "C=C[Cl,Br,I]",
}

def screen_structural_alerts(smiles: str,
                              alert_dict: dict = ICH_M7_ALERTS) -> dict:
    """
    Screen a compound against ICH M7 structural alerts.

    Returns
    -------
    dict with:
      alerts_found : list of triggered alerts
      n_alerts     : count
      ich_m7_class : suggested ICH M7 classification
      smarts_hits  : {alert_name: matching_atom_indices}
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"error": "Invalid SMILES"}

    hits   = {}
    for name, smarts in alert_dict.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt and mol.HasSubstructMatch(patt):
            matches = mol.GetSubstructMatches(patt)
            hits[name] = [list(m) for m in matches]

    # ICH M7 classification logic
    class1_triggers = {k for k in hits if any(
        k in ["Nitrosamine","N_Nitroso_secondary","Nitrosamide","Diazonium"]
    )}
    class2_triggers = {k for k in hits if k not in class1_triggers}

    if class1_triggers:
        ich_class = "Class 1 — Known human mutagen (avoid)"
    elif class2_triggers:
        ich_class = "Class 2 — Known animal mutagen (limit to TTC)"
    elif hits:
        ich_class = "Class 3 — Structural alert, low concern (TTC applies)"
    else:
        ich_class = "Class 5 — No structural alert, no QSAR concern"

    return {
        "alerts_found": list(hits.keys()),
        "n_alerts":     len(hits),
        "ich_m7_class": ich_class,
        "smarts_hits":  hits,
    }

# Screen a test set
test_compounds = [
    ("Aspirin",         "CC(=O)Oc1ccccc1C(=O)O"),
    ("Nitrobenzene",    "O=[N+]([O-])c1ccccc1"),
    ("Aniline",         "Nc1ccccc1"),
    ("N-Nitrosodimethylamine (NDMA)", "CN(C)N=O"),
    ("Furan",           "c1ccoc1"),
    ("Acrolein",        "C=CC=O"),
    ("Caffeine",        "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
]

print(f"{'Compound':35s} {'Alerts':>3} {'ICH M7 Class'}")
print("-" * 85)
for name, smi in test_compounds:
    result = screen_structural_alerts(smi)
    alerts = ", ".join(result["alerts_found"][:3]) if result["alerts_found"] else "None"
    print(f"{name:35s} {result['n_alerts']:>3}   {result['ich_m7_class'][:40]}")
    if result["alerts_found"]:
        print(f"  Alerts: {alerts}")

In [ ]:
# ── 2.2 Two-method ICH M7 integration ────────────────────────────────────────
# ICH M7(R2) Section 3.3: use TWO complementary methods.
# Concordance between methods reduces uncertainty.

def ich_m7_two_method(smiles: str, qsar_model, ad_model) -> dict:
    """
    Apply the ICH M7(R2) two-method framework:
      Method 1: Rule-based structural alerts (expert system)
      Method 2: Statistical QSAR model

    Decision logic per ICH M7(R2):
      Both positive    → Mutagenic concern (Class 2/3)
      Both negative    → No concern (Class 5)
      Discordant       → Expert review needed (Class 3 if SA only)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"error": "Invalid SMILES"}

    # Method 1: structural alerts
    sa_result  = screen_structural_alerts(smiles)
    sa_call    = sa_result["n_alerts"] > 0  # True = alert triggered
    sa_class   = sa_result["ich_m7_class"]

    # Method 2: QSAR
    fp = smiles_to_ecfp4(smiles)
    if fp is None:
        return {"error": "Fingerprint generation failed"}

    in_ad      = ad_model.predict_ad(fp.reshape(1,-1))[0]
    max_tc     = ad_model.get_max_similarity(fp.reshape(1,-1))[0]
    qsar_prob  = qsar_model.predict_proba(fp.reshape(1,-1))[0,1]
    qsar_call  = qsar_prob >= 0.5  # True = predicted mutagenic

    # ICH M7 decision table
    if sa_call and qsar_call:
        decision = "CONCERN"
        rationale = "Both methods flag — mutagenic concern. Requires genotox testing or control."
    elif not sa_call and not qsar_call:
        decision = "NO CONCERN"
        rationale = "Both methods negative — acceptable for ICH M7 Class 5 control."
    elif sa_call and not qsar_call:
        decision = "EQUIVOCAL — EXPERT REVIEW"
        rationale = "Alert triggered but QSAR negative. Assign Class 3; TTC 1.5 μg/day applies."
    else:  # not sa_call and qsar_call
        decision = "EQUIVOCAL — EXPERT REVIEW"
        rationale = "QSAR positive but no alert. QSAR may be outside applicability."

    return {
        "smiles":       smiles,
        "SA_call":      "POSITIVE" if sa_call  else "NEGATIVE",
        "SA_alerts":    sa_result["alerts_found"],
        "QSAR_call":    "POSITIVE" if qsar_call else "NEGATIVE",
        "QSAR_prob":    round(float(qsar_prob), 3),
        "within_AD":    bool(in_ad),
        "max_Tc":       round(float(max_tc), 3),
        "ICH_M7_decision": decision,
        "rationale":    rationale,
    }

print("ICH M7(R2) Two-Method Assessment:")
print("=" * 70)
for name, smi in test_compounds[:5]:
    r = ich_m7_two_method(smi, rf, ad)
    print(f"\n{name}")
    print(f"  SA:   {r['SA_call']:10s}  alerts={r['SA_alerts'][:2]}")
    print(f"  QSAR: {r['QSAR_call']:10s}  prob={r['QSAR_prob']}  AD={r['within_AD']}  maxTc={r['max_Tc']}")
    print(f"  → {r['ICH_M7_decision']}")
    print(f"    {r['rationale'][:70]}")

---
## Section 3 — Read-Across (OECD RAAF)

Read-across is a method to fill data gaps by analogy from source (tested) to target (untested) chemicals. The OECD Read-Across Assessment Framework (RAAF) 2017 provides the regulatory standard.

In [ ]:
# ── 3.1 OECD RAAF category formation ─────────────────────────────────────────
# Categories must be justified by: structural similarity + common mechanism.
# Three types: analogue (1:1), category (many:1), structural fragment.

@dataclass
class ReadAcrossCase:
    """
    OECD RAAF-structured read-across case.
    Maps exactly to IUCLID read-across justification fields.
    """
    target_smiles:     str
    target_name:       str
    sources:           list[dict]         # list of {smiles, name, endpoint_value, reliability}
    endpoint:          str                # e.g. "Acute oral toxicity (LD50)"
    endpoint_units:    str                # e.g. "mg/kg bw"
    mechanism:         str                # why they are similar
    prediction:        Optional[float] = None
    confidence:        Optional[str]   = None
    uncertainty:       list[str]       = field(default_factory=list)

def form_category(target_smiles: str,
                  library_smiles: list[str],
                  library_names:  list[str],
                  similarity_threshold: float = 0.45) -> list[dict]:
    """
    Identify structurally similar source chemicals for read-across.
    Returns list of candidates sorted by Tanimoto similarity.
    """
    target_mol = Chem.MolFromSmiles(target_smiles)
    if target_mol is None:
        return []

    target_fp = AllChem.GetMorganFingerprintAsBitVect(target_mol, 2, 2048)
    candidates = []

    for smi, name in zip(library_smiles, library_names):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp  = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
        tc  = DataStructs.TanimotoSimilarity(target_fp, fp)

        if tc >= similarity_threshold and smi != target_smiles:
            # Compute structural differences
            mw_diff   = abs(Descriptors.MolWt(mol)    - Descriptors.MolWt(target_mol))
            logp_diff = abs(Descriptors.MolLogP(mol)   - Descriptors.MolLogP(target_mol))

            candidates.append({
                "smiles":    smi,
                "name":      name,
                "tanimoto":  round(tc, 4),
                "MW_diff":   round(mw_diff, 1),
                "LogP_diff": round(logp_diff, 2),
            })

    candidates.sort(key=lambda x: x["tanimoto"], reverse=True)
    return candidates

def interpolate_endpoint(sources: list[dict],
                         weights: str = "tanimoto") -> tuple[float, float]:
    """
    Weighted average endpoint from source chemicals.
    Weight by Tanimoto similarity (higher similarity = more weight).
    """
    values = [s["endpoint_value"] for s in sources]
    w      = [s.get("tanimoto", 1.0) for s in sources]
    w_arr  = np.array(w)
    v_arr  = np.array(values)
    w_norm = w_arr / w_arr.sum()
    pred   = float((w_norm * v_arr).sum())
    # Uncertainty: weighted std
    unc    = float(np.sqrt((w_norm * (v_arr - pred)**2).sum()))
    return round(pred, 2), round(unc, 2)

# Demo: read-across for LD50 in a series of phenoxyacetic acids
ACID_LIBRARY = {
    "Acetic acid":               ("CC(=O)O",          5000),  # LD50 mg/kg (rat oral)
    "Phenoxyacetic acid":        ("OCC(=O)Oc1ccccc1",  1300),
    "2-Chlorophenoxyacetic acid":("OCC(=O)Oc1ccccc1Cl", 700),
    "4-Chlorophenoxyacetic acid":("OCC(=O)Oc1ccc(Cl)cc1",800),
    "2,4-Dichlorophenoxyacetic acid (2,4-D)":("OCC(=O)Oc1ccc(Cl)cc1Cl", 375),
}
target_name = "3-Chlorophenoxyacetic acid"
target_smi  = "OCC(=O)Oc1cccc(Cl)c1"  # data gap — no LD50 measured

lib_smi   = [v[0] for v in ACID_LIBRARY.values()]
lib_names = list(ACID_LIBRARY.keys())
lib_ld50  = {k: v[1] for k, v in ACID_LIBRARY.items()}

analogs = form_category(target_smi, lib_smi, lib_names, similarity_threshold=0.30)
print(f"Read-across category for: {target_name}")
print(f"{'Source compound':40s} {'Tc':>6} {'LD50 (mg/kg)':>13}")
print("-" * 62)
for a in analogs:
    ld50 = lib_ld50.get(a["name"], "N/A")
    print(f"{a['name']:40s} {a['tanimoto']:>6.4f} {str(ld50):>13}")

# Interpolate
sources_with_data = [
    {**a, "endpoint_value": lib_ld50[a["name"]]}
    for a in analogs if a["name"] in lib_ld50
]
if sources_with_data:
    pred_ld50, unc = interpolate_endpoint(sources_with_data)
    print(f"\nPredicted LD50 for {target_name}: {pred_ld50} ± {unc} mg/kg bw")
    print(f"GHS category: {'Category 4' if pred_ld50 > 300 else 'Category 3' if pred_ld50 > 50 else 'Category 2'}")

---
## Section 4 — IVIVE: In Vitro to In Vivo Extrapolation

IVIVE converts in vitro effect concentrations (EC50 in cell assay) to in vivo equivalent doses using clearance and bioavailability parameters. The EPA HTTK R package is the regulatory standard.

In [ ]:
# ── 4.1 IVIVE workflow ────────────────────────────────────────────────────────
# EPA HTTK (httk R package / Python port) workflow:
# EC50 (in vitro, μM) → Css_plasma (steady-state conc) → AED (administered equivalent dose)

# Three-compartment IVIVE model (simplified from Rotroff 2010 / Wetmore 2012)

def plasma_protein_binding(logp: float, mw: float) -> float:
    """
    Estimate fraction unbound in plasma (fup) from logP and MW.
    Uses the Lobell & Sivarajah (2003) / Egan (2000) model.
    This is required to convert total concentration to free concentration.
    """
    # Simplified linear model (Obach 2008 approach)
    log_fup = -0.028 * logp - 0.0038 * mw + 1.2
    fup = 10 ** log_fup
    return float(np.clip(fup, 0.001, 1.0))

def hepatic_clearance_estimate(logp: float, mw: float,
                                 n_aro_rings: int) -> float:
    """
    Estimate intrinsic hepatic clearance (CLint, mL/min/mg protein).
    Simplified model — use measured CLint from HLM assay in production.
    """
    # Higher logP → more metabolised; more aromatic rings → more CYP substrate
    clint = 10 ** (0.35*logp + 0.15*n_aro_rings - 0.002*mw + np.random.normal(0, 0.3))
    return float(np.clip(clint, 0.1, 1000.0))

def css_plasma(clint_mLmin_mgprot: float,
               fup: float,
               dose_mg_kg: float = 1.0,
               bw_kg: float = 70.0,
               mppgl: float = 45.0,
               liver_weight_g: float = 1500.0,
               hepatic_blood_flow_mL_min: float = 1500.0) -> float:
    """
    Steady-state plasma concentration using 1-compartment hepatic clearance model.
    (Rotroff 2010, EPA HTTK)

    Css (μM) = dose_rate / (CLh × fup)

    Parameters
    ----------
    clint   : intrinsic clearance (mL/min/mg microsomal protein)
    fup     : fraction unbound in plasma
    dose    : mg/kg/day
    mppgl   : microsomal protein per gram of liver (default 45 mg/g)
    """
    # Liver microsomal CLint → systemic CLh via well-stirred model
    clint_liver = clint_mLmin_mgprot * mppgl * liver_weight_g  # mL/min
    # Well-stirred hepatic clearance
    CLh = (hepatic_blood_flow_mL_min * (clint_liver * fup) /
           (hepatic_blood_flow_mL_min + clint_liver * fup))    # mL/min

    # Dose rate (μmol/min): dose (mg/kg/day) × bw / MW / 1440 min
    # Simplified: use dose in mg/kg, assume MW=300 for generic
    mw_assumed = 300.0
    dose_rate_umol_min = (dose_mg_kg * bw_kg * 1000) / (mw_assumed * 1440)

    Css_total = dose_rate_umol_min / (CLh / 1000)  # μM total
    Css_free  = Css_total * fup                      # μM free (active)
    return Css_free

def aed_from_ec50(ec50_uM: float, logp: float, mw: float,
                   n_aro_rings: int, safety_factor: float = 10.0) -> dict:
    """
    Administered Equivalent Dose (AED) from in vitro EC50.

    Converts EC50_free_in_vitro → Css_target → dose via IVIVE.

    Parameters
    ----------
    ec50_uM      : in vitro EC50 (μM, assumed free/unbound)
    safety_factor: uncertainty factor (default 10 = 10-fold UF)

    Returns NOAEL-equivalent, LOAEL-equivalent, and human AED.
    """
    fup         = plasma_protein_binding(logp, mw)
    clint       = hepatic_clearance_estimate(logp, mw, n_aro_rings)
    target_css  = ec50_uM  # target free Css = in vitro EC50

    # Back-calculate dose from Css target
    clint_liver = clint * 45 * 1500
    CLh = (1500 * clint_liver * fup) / (1500 + clint_liver * fup)
    bw = 70.0
    mw_actual = mw
    dose_aed_mg_kg = (target_css * CLh / 1000 * mw_actual * 1440) / (bw * 1000)

    return {
        "fup":             round(fup, 4),
        "CLint_est":       round(clint, 2),
        "Css_target_uM":   round(target_css, 4),
        "AED_mg_kg_day":   round(dose_aed_mg_kg, 4),
        "NOAEL_equiv":     round(dose_aed_mg_kg / safety_factor, 5),
        "TTC_compare_ug_day": round(dose_aed_mg_kg * 1000 * bw * 1000, 2),  # μg/day at 70 kg
        "TTC_threshold":   1.5,   # μg/day for ICH M7 Class 2/3
    }

# Apply IVIVE to a panel of chemicals
chemicals = [
    ("Bisphenol A",    "CC(C)(c1ccc(O)cc1)c1ccc(O)cc1",    0.5),   # EC50 0.5 μM
    ("Triclosan",      "Oc1ccc(Cl)cc1Oc1cc(Cl)ccc1Cl",     0.1),   # EC50 0.1 μM
    ("Perfluorooctanoic acid", "O=C(O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F", 2.0),
]

print(f"IVIVE Results (EPA HTTK methodology)")
print(f"{'Chemical':28s} {'EC50(μM)':>8} {'fup':>6} {'AED(mg/kg/d)':>14} {'TTC(μg/d)':>12} {'TTC OK?'}")
print("-" * 80)
for name, smi, ec50 in chemicals:
    mol  = Chem.MolFromSmiles(smi)
    if mol:
        logp = Descriptors.MolLogP(mol)
        mw   = Descriptors.MolWt(mol)
        naro = rdMolDescriptors.CalcNumAromaticRings(mol)
        r    = aed_from_ec50(ec50, logp, mw, naro)
        ttc_ok = "✓" if r["TTC_compare_ug_day"] > r["TTC_threshold"] else "✗ FLAG"
        print(f"{name:28s} {ec50:>8.2f} {r['fup']:>6.4f} {r['AED_mg_kg_day']:>14.5f} "
              f"{r['TTC_compare_ug_day']:>12.2f} {ttc_ok}")

---
## Section 5 — Skin Sensitisation Defined Approach (OECD TG 497)

The 2b11 Defined Approach (OECD TG 497) integrates three non-animal methods to predict human skin sensitisation without animal testing.

In [ ]:
# ── 5.1 Skin sensitisation 2o3 Defined Approach ─────────────────────────────
# OECD TG 497 Defined Approach 2 (2o3):
# Input: DPRA (Cys/Lys depletion), KeratinoSens (ARE activation), hCLAT (CD54/CD86)
# Decision: hazard call if ≥ 2 of 3 methods positive

# Simplified mechanistic scoring (replace with in vitro assay data)

def dpra_score(smiles: str) -> dict:
    """
    Direct Peptide Reactivity Assay (DPRA) simulation.
    Measures covalent binding to cysteine and lysine peptides.
    OECD TG 442C (2021 update).
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"call": "INCONCLUSIVE"}

    # Structural features driving DPRA reactivity (simplified)
    michael   = mol.HasSubstructMatch(Chem.MolFromSmarts("[$(C=CC=O),$(C=CS)]"))
    aldehyde  = mol.HasSubstructMatch(Chem.MolFromSmarts("[CHO]"))
    epoxide   = mol.HasSubstructMatch(Chem.MolFromSmarts("C1OC1"))
    acyl_halide=mol.HasSubstructMatch(Chem.MolFromSmarts("C(=O)[Cl,Br]"))
    isocyanate= mol.HasSubstructMatch(Chem.MolFromSmarts("N=C=O"))
    diss_halide=mol.HasSubstructMatch(Chem.MolFromSmarts("[Cl,Br,I][CX4]"))

    reactivity = (0.25*michael + 0.20*aldehyde + 0.20*epoxide +
                  0.15*acyl_halide + 0.15*isocyanate + 0.10*diss_halide)
    reactivity += np.random.normal(0, 0.05)  # assay variability

    cys_depletion = float(np.clip(reactivity * 80, 0, 100))
    lys_depletion = float(np.clip(reactivity * 40, 0, 100))
    # DPRA positive if mean depletion ≥ 6.38% (OECD TG 442C threshold)
    mean_dep = (cys_depletion + lys_depletion) / 2
    call = "POSITIVE" if mean_dep >= 6.38 else "NEGATIVE"
    return {"call": call, "cys_depletion": round(cys_depletion,1),
            "lys_depletion": round(lys_depletion,1), "mean_depletion": round(mean_dep,1)}

def keratino_sens_score(smiles: str) -> dict:
    """
    KeratinoSens ARE-luciferase assay (OECD TG 442D).
    Measures Nrf2/ARE pathway activation in HaCaT keratinocytes.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"call": "INCONCLUSIVE"}

    # Nrf2/ARE activators — electrophiles, oxidants, metal chelators
    michael    = mol.HasSubstructMatch(Chem.MolFromSmarts("[$(C=CC=O)]"))
    quinone    = mol.HasSubstructMatch(Chem.MolFromSmarts("O=c1ccccc1=O"))
    phenol     = mol.HasSubstructMatch(Chem.MolFromSmarts("Oc1ccccc1"))
    isothio    = mol.HasSubstructMatch(Chem.MolFromSmarts("[N]C(=[S])"))

    activation = (0.3*michael + 0.25*quinone + 0.2*phenol + 0.2*isothio)
    activation += np.random.normal(0, 0.05)

    imax = float(np.clip(activation * 200, 1, 300))  # % induction at max conc
    call = "POSITIVE" if imax >= 150 else "NEGATIVE"   # ≥150% induction threshold
    return {"call": call, "imax_pct": round(imax, 1)}

def hclat_score(smiles: str) -> dict:
    """
    Human Cell Line Activation Test (h-CLAT, OECD TG 442E).
    Measures CD54/CD86 upregulation on THP-1 monocytes.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"call": "INCONCLUSIVE"}

    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    # Simplified: cellular sensitisers tend to be mid-MW, moderate logP
    uptake_score = (0.3*(1 < logp < 4) + 0.2*(100 < mw < 500) +
                    0.3*mol.HasSubstructMatch(Chem.MolFromSmarts("[$(C=CC=O),C1OC1,CHO]")))
    uptake_score += np.random.normal(0, 0.08)

    ec3   = float(np.clip(100 * (1 - uptake_score), 0.5, 100))  # μg/mL
    call  = "POSITIVE" if ec3 < 50 else "NEGATIVE"  # EC3 < 50 μg/mL = positive
    return {"call": call, "ec3_ug_mL": round(ec3, 2)}

def sensitisation_2o3(smiles: str, name: str = "") -> dict:
    """
    OECD TG 497 Defined Approach 2:
    Positive if ≥ 2 of 3 methods (DPRA, KeratinoSens, h-CLAT) are positive.
    """
    dpra = dpra_score(smiles)
    ks   = keratino_sens_score(smiles)
    hcl  = hclat_score(smiles)

    n_pos = sum([dpra["call"]=="POSITIVE", ks["call"]=="POSITIVE", hcl["call"]=="POSITIVE"])
    hazard = "SENSITISER" if n_pos >= 2 else "NON-SENSITISER"

    return {
        "compound": name or smiles[:20],
        "DPRA":   dpra["call"], "DPRA_dep": dpra.get("mean_depletion"),
        "KS":     ks["call"],   "KS_imax":  ks.get("imax_pct"),
        "hCLAT":  hcl["call"],  "hCLAT_ec3":hcl.get("ec3_ug_mL"),
        "n_positive": n_pos,
        "hazard_call": hazard,
        "OECD_method": "TG 497 DA 2 (2o3)",
    }

# Panel of known sensitisers and non-sensitisers
skin_panel = [
    ("Cinnamaldehyde (strong sensitiser)",  "O=Cc1ccc(/C=C/c1)C"),
    ("Formaldehyde",                        "C=O"),
    ("1-Chloro-2,4-dinitrobenzene (DNCB)", "O=[N+]([O-])c1ccc(Cl)c([N+](=O)[O-])c1"),
    ("Isopropyl myristate (non-sensitiser)","CCCCCCCCCCCCCC(=O)OC(C)C"),
    ("Lactic acid (non-sensitiser)",        "CC(O)C(=O)O"),
    ("Methylisothiazolinone (sensitiser)",  "Cn1sc(=O)cc1=O"),
]

print("OECD TG 497 Skin Sensitisation Defined Approach (2o3)")
print(f"{'Compound':38s} {'DPRA':>8} {'KS':>8} {'hCLAT':>8} {'n+':>3} {'Hazard'}")
print("-" * 80)
np.random.seed(0)  # for reproducibility of simulated assay noise
for name, smi in skin_panel:
    r = sensitisation_2o3(smi, name)
    print(f"{name[:38]:38s} {r['DPRA']:>8} {r['KS']:>8} {r['hCLAT']:>8} "
          f"{r['n_positive']:>3}  {r['hazard_call']}")

---
## Section 6 — CiPA Cardiac Safety (Multi-Channel + hiPS-CM)

The CiPA (Comprehensive In vitro Proarrhythmia Assay) paradigm replaces hERG-only testing with a mechanistic multi-channel assessment of TdP (Torsades de Pointes) risk.

In [ ]:
# ── 6.1 CiPA multi-channel IC50 assessment ───────────────────────────────────
# CiPA integrates IC50 data for 7 cardiac ion channels into an in silico
# AP (action potential) model to predict net proarrhythmic risk.

# Seven CiPA channels and their roles
CIPA_CHANNELS = {
    "IKr  (hERG)":   {"risk_direction": "block_pro",  "weight": 0.40},  # main proarrhythmic
    "INaL (Nav1.5)":  {"risk_direction": "block_pro",  "weight": 0.15},
    "ICaL (Cav1.2)":  {"risk_direction": "block_anti", "weight": 0.20},  # block is protective
    "IKs  (KCNQ1)":   {"risk_direction": "block_pro",  "weight": 0.10},
    "INaF (Nav1.5)":  {"risk_direction": "block_pro",  "weight": 0.05},
    "If   (HCN4)":    {"risk_direction": "neutral",    "weight": 0.05},
    "Ito  (Kv4.3)":   {"risk_direction": "neutral",    "weight": 0.05},
}

def simulate_cipa_ic50(smiles: str, seed: int = 42) -> dict:
    """
    Simulate multi-channel IC50 data (replace with measured patch-clamp data).
    Real CiPA workflow: automated patch clamp (QPatch, IonWorks, SyncroPatch).
    """
    np.random.seed(seed)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {}

    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    basic_n = sum(1 for a in mol.GetAtoms()
                  if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    ar_rings = rdMolDescriptors.CalcNumAromaticRings(mol)

    # hERG IC50 driven by logP, basic N, aromaticity
    herg_ic50 = 10 ** (-(0.3*logp + 0.2*basic_n + 0.1*ar_rings - 1.5)
                        + np.random.normal(0, 0.4))
    ic50_data = {"IKr  (hERG)": herg_ic50}

    # Other channels: correlated but with individual variability
    ic50_data["INaL (Nav1.5)"]  = herg_ic50 * np.random.uniform(2, 8)
    ic50_data["ICaL (Cav1.2)"]  = herg_ic50 * np.random.uniform(5, 20)
    ic50_data["IKs  (KCNQ1)"]   = herg_ic50 * np.random.uniform(3, 15)
    ic50_data["INaF (Nav1.5)"]  = herg_ic50 * np.random.uniform(1, 5)
    ic50_data["If   (HCN4)"]    = herg_ic50 * np.random.uniform(10, 100)
    ic50_data["Ito  (Kv4.3)"]   = herg_ic50 * np.random.uniform(8, 50)

    return {k: round(float(v), 3) for k, v in ic50_data.items()}

def cipa_risk_score(ic50_dict: dict,
                    therapeutic_conc_uM: float = 0.1) -> dict:
    """
    CiPA net risk score using the Fermini (2016) / Dutta (2017) framework.

    Risk score = Σ w_i × occupancy_i × direction_i
    where occupancy = 1 / (1 + IC50/(free_Cmax))
    """
    Cmax = therapeutic_conc_uM  # free (unbound) Cmax

    net_risk = 0.0
    channel_scores = {}

    for channel, props in CIPA_CHANNELS.items():
        ic50 = ic50_dict.get(channel, float("inf"))
        # Channel occupancy at Cmax
        occupancy = 1 / (1 + ic50 / Cmax) if Cmax > 0 else 0

        if props["risk_direction"] == "block_pro":
            contribution = props["weight"] * occupancy      # pro-arrhythmic
        elif props["risk_direction"] == "block_anti":
            contribution = -props["weight"] * occupancy     # anti-arrhythmic
        else:
            contribution = 0

        net_risk += contribution
        channel_scores[channel] = {
            "IC50_uM":     round(ic50, 2),
            "occupancy":   round(occupancy, 4),
            "contribution":round(contribution, 5),
            "direction":   props["risk_direction"],
        }

    # TdP risk category (based on CiPA classification)
    if net_risk > 0.15:   risk_cat = "HIGH TdP risk"
    elif net_risk > 0.05: risk_cat = "INTERMEDIATE TdP risk"
    else:                 risk_cat = "LOW TdP risk"

    return {
        "cipa_net_score": round(net_risk, 4),
        "tdp_category":   risk_cat,
        "free_cmax_uM":   Cmax,
        "channel_scores": channel_scores,
    }

# Panel of drugs with known TdP risk
cipa_panel = [
    ("Cisapride (HIGH TdP)", "COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1", 0.002),
    ("Moxifloxacin (LOW)",   "COc1c2n(cc(C(=O)O)c(=O)c2c(F)cc1N1CC2CCCC2C1)C1CC1", 0.5),
    ("Verapamil (ANTI)",     "COc1ccc(CCN(C)CCCC(C#N)(c2ccc(OC)c(OC)c2)C(C)C)cc1OC", 0.1),
    ("Caffeine (MINIMAL)",   "Cn1cnc2c1c(=O)n(C)c(=O)n2C", 10.0),
]

print("CiPA Multi-Channel Cardiac Safety Assessment")
print("="*70)
for name, smi, cmax in cipa_panel:
    ic50s = simulate_cipa_ic50(smi, seed=hash(name)%1000)
    result = cipa_risk_score(ic50s, therapeutic_conc_uM=cmax)
    print(f"\n{name}")
    print(f"  Free Cmax: {cmax} μM")
    print(f"  hERG IC50: {ic50s.get('IKr  (hERG)', 'N/A')} μM")
    print(f"  CiPA net score: {result['cipa_net_score']:.4f}")
    print(f"  → {result['tdp_category']}")

---
## Section 7 — PBPK-NAM Integration

Physiologically-Based Pharmacokinetic (PBPK) modelling connects in vitro data to in vivo concentrations — the dosimetry bridge between NAMs and regulatory thresholds.

In [ ]:
# ── 7.1 Minimal PBPK model for NAM dosimetry ─────────────────────────────────
# A 3-compartment PBPK: gut, liver (metabolising), systemic.
# Used by EPA (HTTK), EFSA (OpenFoodTox), FDA (PBPK guidance 2018).

from scipy.integrate import solve_ivp

def minimal_pbpk(dose_mg_kg: float, mw: float,
                 logp: float, fup: float,
                 clint_mL_min_mg: float = 10.0,
                 fa: float = 0.90,
                 t_end_h: float = 24.0,
                 n_timepoints: int = 200) -> dict:
    """
    3-compartment PBPK for oral dosing.
    Compartments: gut (absorption) → liver (metabolism) → central (plasma)

    Parameters
    ----------
    dose_mg_kg   : oral dose (mg/kg body weight)
    mw           : molecular weight (g/mol)
    logp         : partition coefficient (lipophilicity)
    fup          : fraction unbound in plasma
    clint        : intrinsic hepatic clearance (mL/min/mg microsomal protein)
    fa           : fraction absorbed from gut (default 0.90)

    Returns dict with Cmax, AUC, t_half, Css time series.
    """
    # Physiological parameters (70 kg human, ICRP 2002)
    BW          = 70.0      # body weight (kg)
    Vplasma     = 3.0       # plasma volume (L)
    Vliver      = 1.5       # liver volume (L)
    Qh          = 90.0      # hepatic blood flow (L/h)
    MPPGL       = 45.0      # mg microsomal protein per gram liver
    liver_g     = 1500.0    # liver mass (g)

    # Distribution volume (Vd) from logP — Berezhkovskiy 2004 approximation
    Vd = max(0.5, 0.2 + 0.8 * logp) * BW  # L

    # Hepatic intrinsic clearance
    CLint_liver = clint_mL_min_mg * MPPGL * liver_g / 1000  # L/h
    # Well-stirred hepatic clearance
    CLh = (Qh * CLint_liver * fup) / (Qh + CLint_liver * fup)  # L/h

    # Initial dose in plasma (mg → μmol)
    dose_umol = (dose_mg_kg * BW * 1000 / mw) * fa * 1e3  # nmol → adjust for μM

    # ODE system: 1-compartment with first-order elimination
    # dA/dt = -CLh/Vd * A  (simplified from 3-compartment for demo)
    k_elim = CLh / Vd  # h⁻¹

    t_eval = np.linspace(0, t_end_h, n_timepoints)
    # Analytical solution: C(t) = C0 * exp(-k * t)
    C0    = dose_umol / (Vd * 1000)  # μM
    Ct    = C0 * np.exp(-k_elim * t_eval)
    Ct_free = Ct * fup

    # Key PK parameters
    Cmax     = float(C0)
    Cmax_free= float(Cmax * fup)
    t_half   = float(np.log(2) / k_elim)
    AUC      = float(C0 / k_elim)  # μM·h (analytical)

    return {
        "Cmax_total_uM":   round(Cmax, 4),
        "Cmax_free_uM":    round(Cmax_free, 4),
        "t_half_h":        round(t_half, 2),
        "AUC_uM_h":        round(AUC, 2),
        "Vd_L":            round(Vd, 1),
        "CLh_L_h":         round(CLh, 2),
        "fup":             round(fup, 4),
        "t_eval":          t_eval,
        "Ct_total":        Ct,
        "Ct_free":         Ct_free,
    }

# Run PBPK for two compounds
compounds = [
    ("Bisphenol A", "CC(C)(c1ccc(O)cc1)c1ccc(O)cc1", 50.0,  0.005, 15.0),
    ("Ibuprofen",   "CC(C)Cc1ccc(cc1)C(C)C(=O)O",    400.0, 0.010, 0.2),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, smi, dose, fup_v, clint_v) in zip(axes, compounds):
    mol  = Chem.MolFromSmiles(smi)
    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    pk   = minimal_pbpk(dose, mw, logp, fup_v, clint_mL_min_mg=clint_v)

    ax.semilogy(pk["t_eval"], pk["Ct_total"], '#1565C0', lw=2.5, label='Total plasma')
    ax.semilogy(pk["t_eval"], pk["Ct_free"],  '#E74C3C', lw=2.0, linestyle='--', label='Free (unbound)')
    ax.set_xlabel('Time (h)'); ax.set_ylabel('Concentration (μM)')
    ax.set_title(f'{name} — PBPK\nCmax={pk["Cmax_total_uM"]:.3f} μM  t½={pk["t_half_h"]:.1f}h', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    print(f"{name}: Cmax={pk['Cmax_total_uM']:.4f} μM total  |  {pk['Cmax_free_uM']:.4f} μM free  |  t½={pk['t_half_h']:.1f}h")

plt.tight_layout(); plt.show()

---
## Section 8 — Adverse Outcome Pathway (AOP) Framework

AOPs connect a **Molecular Initiating Event (MIE)** through **Key Events (KEs)** to an **Adverse Outcome (AO)** at the level of regulatory concern. OECD AOP-Wiki hosts 400+ AOPs.

In [ ]:
# ── 8.1 AOP data structure and scoring ───────────────────────────────────────
# AOP #54: Inhibition of Na+/I- symporter → thyroid hormone synthesis decrease → hypothyroidism

@dataclass
class KeyEvent:
    ke_id:       int
    name:        str
    level:       str    # molecular, cellular, tissue, organ, individual, population
    assay:       str    # how it is measured
    threshold:   float
    score:       Optional[float] = None  # measured value

@dataclass
class AOP:
    aop_id:  int
    title:   str
    mie:     KeyEvent
    kes:     list[KeyEvent]
    ao:      KeyEvent
    stressor_smarts: list[str]  # SMARTS to identify chemical stressors

    def score_ke(self, ke_name: str, value: float):
        for ke in [self.mie] + self.kes + [self.ao]:
            if ke.name == ke_name:
                ke.score = value

    def weight_of_evidence(self) -> dict:
        """Simplified KER (Key Event Relationship) weight-of-evidence score."""        all_kes = [self.mie] + self.kes + [self.ao]
        scored  = [ke for ke in all_kes if ke.score is not None]
        if not scored:
            return {"woe": 0.0, "confidence": "insufficient data"}
        above_threshold = sum(1 for ke in scored if ke.score > ke.threshold)
        woe = above_threshold / len(all_kes)
        conf = "HIGH" if woe > 0.7 else "MODERATE" if woe > 0.4 else "LOW"
        return {"woe": round(woe, 2), "confidence": conf,
                "n_triggered": above_threshold, "n_total": len(all_kes)}

# Build AOP #54 (thyroid) and AOP #8 (aromatase inhibition)
aop_thyroid = AOP(
    aop_id=54,
    title="Inhibition of Na+/I- Symporter (NIS) and Thyroid Hormone Synthesis",
    mie=KeyEvent(1, "NIS inhibition", "molecular", "Fluorescence-based NIS assay", threshold=30.0),
    kes=[
        KeyEvent(2, "Thyroid iodide uptake decrease", "cellular", "131I uptake (FRTL-5)", threshold=25.0),
        KeyEvent(3, "Thyroglobulin iodination decrease", "tissue", "TG iodination assay", threshold=20.0),
        KeyEvent(4, "T3/T4 hormone synthesis decrease", "tissue", "ELISA T3/T4", threshold=15.0),
        KeyEvent(5, "TSH increase (feedback)", "organ", "TSH ELISA", threshold=10.0),
    ],
    ao=KeyEvent(6, "Hypothyroidism / developmental neurotoxicity", "individual",
                "Clinical/rodent thyroid function tests", threshold=1.0),
    stressor_smarts=["[#53]CC",           # iodoalkane
                     "OC(=O)c1ccc(I)cc1", # iodobenzoic acids
                     "[Cl,F,Br]c1ccccc1S"],# halogenated phenols
)

# Score with simulated assay data for perchlorate (NIS inhibitor)
np.random.seed(42)
aop_thyroid.score_ke("NIS inhibition", 65.0)               # 65% inhibition
aop_thyroid.score_ke("Thyroid iodide uptake decrease", 55.0)
aop_thyroid.score_ke("Thyroglobulin iodination decrease", 8.0)  # below threshold
aop_thyroid.score_ke("T3/T4 hormone synthesis decrease", 18.0)
aop_thyroid.score_ke("TSH increase (feedback)", 12.0)
aop_thyroid.score_ke("Hypothyroidism / developmental neurotoxicity", 0.5)  # not yet observed

woe = aop_thyroid.weight_of_evidence()
print(f"AOP #{aop_thyroid.aop_id}: {aop_thyroid.title}")
print(f"Weight of Evidence: {woe['woe']:.0%}  ({woe['n_triggered']}/{woe['n_total']} KEs triggered)")
print(f"Confidence: {woe['confidence']}")
print()

all_kes = [aop_thyroid.mie] + aop_thyroid.kes + [aop_thyroid.ao]
for ke in all_kes:
    if ke.score is not None:
        triggered = "✓" if ke.score > ke.threshold else "✗"
        print(f"  {triggered} KE{ke.ke_id}: {ke.name[:45]:45s} score={ke.score:.1f}  threshold={ke.threshold}")

---
## Section 9 — IATA: Integrated Testing & Assessment Approach

In [ ]:
# ── 9.1 Full IATA weight-of-evidence engine ──────────────────────────────────
# OECD GD 255 (2023): IATA integrates all available data (QSAR, in vitro,
# read-across, in vivo, AOP) into a structured, transparent hazard conclusion.

@dataclass
class IATAEvidence:
    source:      str       # e.g. "QSAR (RF-ECFP4)", "DPRA", "ICH M7 SA"
    endpoint:    str       # e.g. "Skin sensitisation", "Ames mutagenicity"
    call:        str       # "POSITIVE", "NEGATIVE", "INCONCLUSIVE"
    confidence:  str       # "HIGH", "MODERATE", "LOW"
    reliability: int       # Klimisch 1–4 (1=reliable without restriction)
    within_ad:   bool = True
    notes:       str = ""

def iata_weight_of_evidence(smiles: str, compound_name: str,
                              evidence_list: list[IATAEvidence]) -> dict:
    """
    OECD GD 255-aligned IATA WoE assessment.

    Aggregation rules:
      - Reliability 1-2 evidence weighted more
      - Out-of-AD predictions downweighted
      - Majority call from weighted votes
      - Overall confidence from evidence density
    """
    weights = {"HIGH": 1.0, "MODERATE": 0.6, "LOW": 0.3}
    reliability_weights = {1: 1.0, 2: 0.8, 3: 0.5, 4: 0.2}

    pos_weight = 0.0
    neg_weight = 0.0
    total_weight = 0.0

    for ev in evidence_list:
        w = weights.get(ev.confidence, 0.3) * reliability_weights.get(ev.reliability, 0.5)
        if not ev.within_ad:
            w *= 0.4  # strongly downweight out-of-AD predictions
        if ev.call == "POSITIVE":
            pos_weight += w
        elif ev.call == "NEGATIVE":
            neg_weight += w
        total_weight += w

    if total_weight == 0:
        return {"conclusion": "INCONCLUSIVE", "confidence": "INSUFFICIENT DATA"}

    pos_frac = pos_weight / total_weight
    neg_frac = neg_weight / total_weight

    if pos_frac >= 0.6:
        conclusion = "HAZARD IDENTIFIED"
    elif neg_frac >= 0.6:
        conclusion = "NO HAZARD IDENTIFIED"
    else:
        conclusion = "EQUIVOCAL — ADDITIONAL DATA NEEDED"

    overall_conf = ("HIGH"     if len(evidence_list) >= 4 and total_weight > 2.5
               else "MODERATE" if len(evidence_list) >= 2
               else "LOW")

    return {
        "compound":      compound_name,
        "smiles":        smiles,
        "conclusion":    conclusion,
        "confidence":    overall_conf,
        "pos_weight":    round(pos_weight, 3),
        "neg_weight":    round(neg_weight, 3),
        "pos_fraction":  round(pos_frac, 3),
        "n_evidence":    len(evidence_list),
        "evidence_summary": [f"{e.source}: {e.call} (rel={e.reliability})"
                              for e in evidence_list],
    }

# Demo: cinnamaldehyde skin sensitisation IATA
cinn_evidence = [
    IATAEvidence("DEREK Nexus (SA)", "Skin sensitisation", "POSITIVE", "HIGH", 1,
                 notes="alpha-beta unsaturated aldehyde alert"),
    IATAEvidence("DPRA (OECD TG 442C)", "Skin sensitisation", "POSITIVE", "HIGH", 1,
                 notes="Cys depletion 78%, Lys depletion 42%"),
    IATAEvidence("KeratinoSens (OECD TG 442D)", "Skin sensitisation", "POSITIVE", "HIGH", 1,
                 notes="iMax 240% at 100 μM"),
    IATAEvidence("h-CLAT (OECD TG 442E)", "Skin sensitisation", "POSITIVE", "HIGH", 1,
                 notes="CD54 EC3 = 6 μg/mL"),
    IATAEvidence("LLNA (OECD TG 429)", "Skin sensitisation", "POSITIVE", "HIGH", 1,
                 notes="EC3 = 0.07% (extreme sensitiser)", reliability=1),
]

result = iata_weight_of_evidence(
    "O=Cc1ccc(/C=C/c1)C", "Cinnamaldehyde", cinn_evidence
)
print("IATA Weight-of-Evidence Report")
print("=" * 65)
print(f"Compound: {result['compound']}")
print(f"Conclusion: {result['conclusion']}")
print(f"Overall confidence: {result['confidence']}")
print(f"Evidence weight: POSITIVE={result['pos_weight']:.2f}  NEGATIVE={result['neg_weight']:.2f}")
print(f"\nEvidence sources ({result['n_evidence']}):")
for ev_str in result["evidence_summary"]:
    print(f"  • {ev_str}")

---
## Section 10 — Regulatory Report Generation

In [ ]:
# ── 10.1 Machine-readable regulatory summary ─────────────────────────────────
# Generate a structured NAM assessment package in regulatory format.
# Follows: EPA SAB NAM report template, ECHA R.7 format, FDA FIH summary.

def generate_nam_report(smiles: str, compound_name: str,
                         cas: str = "N/A") -> dict:
    """
    Generate a complete NAM toxicology package for a compound.
    Returns dict structured for EPA/FDA/ECHA submission.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"error": "Invalid SMILES"}

    # Physicochemical profile
    mw    = Descriptors.MolWt(mol)
    logp  = Descriptors.MolLogP(mol)
    tpsa  = Descriptors.TPSA(mol)
    hbd   = rdMolDescriptors.CalcNumHBD(mol)
    hba   = rdMolDescriptors.CalcNumHBA(mol)
    qed   = QED.qed(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    inchi_key = Chem.inchi.InchiToInchiKey(Chem.inchi.MolToInchi(mol))

    # Run all NAM modules
    np.random.seed(42)
    sa_result = screen_structural_alerts(smiles)
    fp = smiles_to_ecfp4(smiles)
    in_ad    = bool(ad.predict_ad(fp.reshape(1,-1))[0]) if fp is not None else False
    qsar_prob = float(rf.predict_proba(fp.reshape(1,-1))[0,1]) if fp is not None else None
    skin_sens = sensitisation_2o3(smiles, compound_name)
    fup_est   = plasma_protein_binding(logp, mw)
    clint_est = hepatic_clearance_estimate(logp, mw, rdMolDescriptors.CalcNumAromaticRings(mol))
    ivive_r   = aed_from_ec50(0.5, logp, mw, rdMolDescriptors.CalcNumAromaticRings(mol))

    # Assemble report
    report = {
        "report_type":  "NAM Toxicology Assessment Package",
        "version":      "1.0",
        "guidelines":   ["ICH M7(R2)", "OECD TG 497", "EPA HTTK", "OECD GD 255"],

        "compound_identification": {
            "name":      compound_name,
            "cas":       cas,
            "smiles":    smiles,
            "inchikey":  inchi_key,
            "formula":   formula,
        },

        "physicochemical_profile": {
            "MW_Da": round(mw, 2),    "LogP": round(logp, 3),
            "TPSA_A2": round(tpsa,1), "HBD": hbd, "HBA": hba,
            "QED": round(qed, 3),
            "Ro5_violations": sum([mw>500, logp>5, hbd>5, hba>10]),
        },

        "genotoxicity": {
            "method": "ICH M7(R2) two-method framework",
            "SA_call":     "POSITIVE" if sa_result["n_alerts"] > 0 else "NEGATIVE",
            "SA_alerts":   sa_result["alerts_found"],
            "QSAR_call":   ("POSITIVE" if qsar_prob and qsar_prob >= 0.5 else "NEGATIVE"),
            "QSAR_prob":   round(qsar_prob, 3) if qsar_prob else None,
            "within_AD":   in_ad,
            "conclusion":  sa_result["ich_m7_class"],
        },

        "skin_sensitisation": {
            "method":   "OECD TG 497 DA2 (2o3)",
            "DPRA":     skin_sens["DPRA"],
            "KeratinoSens": skin_sens["KS"],
            "hCLAT":    skin_sens["hCLAT"],
            "hazard_call": skin_sens["hazard_call"],
        },

        "toxicokinetics": {
            "method":   "EPA HTTK IVIVE (3-compartment)",
            "fup":      round(fup_est, 4),
            "CLint_mL_min_mg": round(clint_est, 2),
            "AED_mg_kg_day": ivive_r["AED_mg_kg_day"],
            "NOAEL_equiv_mg_kg_day": ivive_r["NOAEL_equiv"],
            "TTC_comparison_ug_day": ivive_r["TTC_compare_ug_day"],
            "TTC_flag": ivive_r["TTC_compare_ug_day"] < ivive_r["TTC_threshold"],
        },

        "overall_conclusion": {
            "approach": "Tiered IATA per OECD GD 255",
            "key_concerns": (
                ([f"Genotox alert: {sa_result['ich_m7_class']}"]
                 if sa_result["n_alerts"] > 0 else []) +
                (["Skin sensitisation hazard identified"]
                 if skin_sens["hazard_call"] == "SENSITISER" else []) +
                (["TTC exceedance — additional ADME data required"]
                 if ivive_r["TTC_compare_ug_day"] < 1.5 else [])
            ),
            "recommended_next_step": (
                "No further testing required" if (
                    sa_result["n_alerts"] == 0 and
                    skin_sens["hazard_call"] != "SENSITISER" and
                    ivive_r["TTC_compare_ug_day"] >= 1.5
                ) else "Additional in vitro testing / expert review required"
            ),
        }
    }
    return report

# Generate report for Cinnamaldehyde
report = generate_nam_report("O=Cc1ccc(/C=C/c1)C", "trans-Cinnamaldehyde", "104-55-2")

print("NAM ASSESSMENT PACKAGE")
print("=" * 60)
print(json.dumps({
    k: v for k, v in report.items()
    if k not in ("physicochemical_profile",)  # skip for brevity
}, indent=2)[:2500])
print("...")
print(f"\nKey concerns: {report['overall_conclusion']['key_concerns']}")
print(f"Recommendation: {report['overall_conclusion']['recommended_next_step']}")

In [ ]:
# ── 10.2 Regulatory cheatsheet ───────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                New Approach Methods — Regulatory Reference              ║
╠══════════════════════════════════════════════════════════════════════════╣
║ GENOTOXICITY / MUTAGENICITY                                              ║
║  ICH M7(R2) 2023  Two complementary in silico methods required          ║
║  Method 1:        Expert/rule-based structural alerts (DEREK, SARAH)    ║
║  Method 2:        Statistical QSAR (ECFP4 + RF/XGBoost/DNN)            ║
║  Sensitivity:     ≥ 90% required for each method                        ║
║  AD:              Must flag out-of-domain compounds                     ║
║  TTC threshold:   1.5 μg/day (Class 2/3), 0.0025 μg/day (Class 1)     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ SKIN SENSITISATION                                                       ║
║  OECD TG 497 DA2: 2o3 (DPRA + KeratinoSens + hCLAT) — validated 2023  ║
║  OECD TG 497 DA3: DASS — uses same 3 assays, probabilistic output      ║
║  No LLNA required if DA gives unambiguous result                        ║
║  Potency (GHS 1A/1B): requires DPRA Cys% + EC3 data                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║ CARDIAC SAFETY                                                           ║
║  ICH E14/S7B 2022  CiPA replaces hERG-only paradigm                    ║
║  Required:         Multi-channel IC50 (IKr, INaL, ICaL, IKs)           ║
║  In silico:        O'Hara-Rudy AP model, CiPA risk score               ║
║  hiPS-CM:          Stem cell cardiomyocyte confirmatory assay           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ IVIVE / PBPK                                                             ║
║  EPA HTTK:         3-compartment, fup + CLint → Css → AED              ║
║  FDA PBPK (2018):  Accepted for FIH dose selection, DDI, renal/hepatic  ║
║  EFSA (2021):      PBPK for pesticide dietary exposure                  ║
║  Key inputs:       fup (equilibrium dialysis), CLint (HLM/HEP), Papp   ║
╠══════════════════════════════════════════════════════════════════════════╣
║ READ-ACROSS                                                              ║
║  OECD RAAF (2017): Scenario 1–6, source adequacy scoring               ║
║  IUCLID:           Formal read-across report with uncertainty analysis  ║
║  Key justification: structural similarity + common mechanism + ADME     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ IATA / WOE                                                               ║
║  OECD GD 255:      Structured IATA template                            ║
║  ECHA R.7:         Endpoint-specific weight of evidence                ║
║  Klimisch score:   1=reliable, 2=reliable with restriction, 3=not rel. ║
║  Key principle:    Convergent evidence > single data point             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ APPLICABILITY DOMAIN                                                     ║
║  OECD GD 69:       Minimum AD requirements                             ║
║  Tanimoto-based:   max(Tc_to_training) ≥ threshold                     ║
║  Leverage:         h* = 3k/n  (Williams plot)                          ║
║  Bounding box:     descriptor range of training set                    ║
╚══════════════════════════════════════════════════════════════════════════╝
""")